# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bander03/FlyRank_Intern/blob/main/work/notebooks/capstone.ipynb)

This notebook re-derives, end to end, every number and figure the deployed paper (`docs/index.html`)
shows — running it top to bottom regenerates the paper's evidence from scratch, on the same
anonymized starter dataset used throughout the internship.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

**Research question:** Do structural and visibility signals — search position, impression
volume, content type, word count, age, freshness, and engagement — predict which pages have a
CTR that underperforms for their own content type, without reading the CTR value itself?

**The decision this supports:** which pages a content reviewer should look at first when
triaging a large backlog for a possible title/meta or content refresh. Out of thousands of
pages, the valuable question is *which one to check first* — a ranking problem on real,
messy search data, exactly where a learned model can beat a fixed rule (see
`skills/flyrank/flyrank-context/SKILL.md`).

**Why "without reading CTR" matters:** a rule that flags low CTR by directly reading CTR is a
threshold, not a discovery. The interesting, useful version of this question is whether *other*
signals — available without waiting for enough click data to trust a raw CTR number — carry real
predictive information on their own.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** the FlyRank ML Internship starter dataset, `data/raw/content_refresh_anonymized.csv`
— 30,000 rows × 44 columns, one row per pseudonymized content item, 32 pseudonymized clients,
metrics aggregated over a trailing 90-day window ending at export time. No client names, URLs,
titles, or keywords are present anywhere in this file — it ships pre-anonymized.

**Filter applied:** `impressions_90d >= 100`, the same visibility floor used throughout this
project (down to 22,006 rows) — pages with negligible search visibility don't have a trustworthy
CTR signal to evaluate in the first place.

**Excluded, and why:**
- `ctr` — deliberately excluded from the model's features. It defines the label this paper
  predicts; including it would make the "model" a lookup table, not a discovery.
- `trend_pct`, `trend_direction` — the dataset's other label-derived trap (they define a
  *different* label, `is_declining_label`, used elsewhere in this repo's reference pipeline).
  Not relevant to this paper's label, and confirmed absent from the feature set regardless.
- `content_id`, `client_id` — pseudonymous identifiers, used only for grouping the train/test
  split, never as model inputs.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label — `low_ctr_for_type`:** 1 when a page's CTR sits at or below the 40th percentile of
CTR for its own `content_type`, threshold learned from the training split only. CTR is heavily
zero-inflated (27-72% of rows are exactly 0, depending on type), so the 25th percentile is 0 for
every type and a strict "below" test never fires — the 40th percentile with "at or below" is the
lowest cut that actually separates rows given that shape.

**Features (7 numeric + 1 categorical):** `avg_position`, `impressions_90d`, `word_count`,
`content_age_days`, `days_since_last_update`, `engagement_rate`, `scroll_rate`, `content_type`.
`ctr` excluded by design (see Data).

**Baseline:** a transparent hand-rule — `visibility_score x position_reachable_gate x
ctr_gap_vs_own_type_median` — scored and ranked with one reason code, one action threshold.
Because the baseline is a rule, not a fitted model, it's allowed to read `ctr` directly; the
model is not (see Results for why that makes the baseline's own precision less impressive than
it first looks).

**Validation design — grouped by client, not random:** a random row-level split lets pages from
the same client appear in both train and test, letting a model partly memorize client-level
patterns rather than learn something that generalizes. Every split in this paper is
`GroupShuffleSplit` grouped by `client_id`, 75/25, seed 42.

**Leakage checks:** (1) train-without/train-with test — adding `ctr` back as a feature and
confirming precision jumps to a suspiciously perfect 1.00 at every K, then removing it and
confirming the honest number returns; (2) split honesty test — comparing a naive random split
against the grouped split and checking client overlap between train and test directly, not just
the resulting metric. Both run fresh below.

In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

RANDOM_SEED = 42
pd.set_option("display.width", 120)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
active = df[df["impressions_90d"] >= 100].copy().reset_index(drop=True)
print(f"Active rows (impressions_90d >= 100): {len(active):,}")

num_feats = ["avg_position", "impressions_90d", "word_count", "content_age_days",
             "days_since_last_update", "engagement_rate", "scroll_rate"]
cat_feats = ["content_type"]
for f in num_feats:
    active[f] = active[f].fillna(0)
active["content_type"] = active["content_type"].fillna("unknown")

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def make_pre():
    return ColumnTransformer([("num", "passthrough", num_feats),
                               ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats)])

# ---- Honest client-grouped split (used for every headline number below) ----
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
tr_idx, te_idx = next(gss.split(active, groups=active["client_id"]))
train, test = active.iloc[tr_idx].copy(), active.iloc[te_idx].copy()

train_type_p40 = train.groupby("content_type")["ctr"].quantile(0.40)
global_p40 = train["ctr"].quantile(0.40)
def label_low_ctr(frame):
    thr = frame["content_type"].map(train_type_p40).fillna(global_p40)
    return (frame["ctr"] <= thr).astype(int)
train["label"] = label_low_ctr(train)
test["label"] = label_low_ctr(test)

print(f"Train: {len(train):,} rows, {train['client_id'].nunique()} clients")
print(f"Test:  {len(test):,} rows, {test['client_id'].nunique()} clients")
print(f"Client overlap: {len(set(train['client_id']) & set(test['client_id']))}")
print(f"Base rate (test): {test['label'].mean():.3f}")

Active rows (impressions_90d >= 100): 22,006
Train: 17,396 rows, 22 clients
Test:  4,610 rows, 8 clients
Client overlap: 0
Base rate (test): 0.399


In [2]:
# ---- Baseline (reads ctr directly, hand rule) ----
def baseline_score(frame):
    type_median_ctr = frame.groupby("content_type")["ctr"].transform("median")
    ctr_gap = (type_median_ctr - frame["ctr"]).clip(lower=0)
    ctr_gap_norm = (ctr_gap / type_median_ctr.replace(0, np.nan)).fillna(0).clip(0, 1)
    position_ok = ((frame["avg_position"] > 0) & (frame["avg_position"] <= 20)).astype(int)
    visibility_score = frame["impressions_90d"].rank(pct=True)
    return visibility_score * position_ok * ctr_gap_norm

active["baseline_score"] = baseline_score(active)
test_baseline_score = active.loc[test.index, "baseline_score"]

# ---- Logistic Regression + Random Forest, ctr excluded ----
lr = Pipeline([("pre", make_pre()), ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_SEED))])
lr.fit(train[num_feats + cat_feats], train["label"])
lr_proba = lr.predict_proba(test[num_feats + cat_feats])[:, 1]

rf = Pipeline([("pre", make_pre()), ("clf", RandomForestClassifier(
    n_estimators=300, min_samples_leaf=5, random_state=RANDOM_SEED))])
rf.fit(train[num_feats + cat_feats], train["label"])
rf_proba = rf.predict_proba(test[num_feats + cat_feats])[:, 1]

base_rate = float(test["label"].mean())
rows = []
for name, scores in [("baseline (reads ctr directly)", test_baseline_score.values),
                      ("logistic_regression (no ctr)", lr_proba),
                      ("random_forest (no ctr)", rf_proba)]:
    row = {"model": name}
    for k in (20, 50, 100):
        row[f"precision@{k}"] = round(float(precision_at_k(scores, test["label"].values, k)), 3)
    rows.append(row)
comparison_table = pd.DataFrame(rows)
comparison_table["base_rate"] = round(base_rate, 3)
print(comparison_table.to_string(index=False))

                        model  precision@20  precision@50  precision@100  base_rate
baseline (reads ctr directly)          1.00          0.96           0.96      0.399
 logistic_regression (no ctr)          0.95          0.92           0.88      0.399
       random_forest (no ctr)          1.00          0.90           0.83      0.399


In [3]:
# ---- Leakage confession test: add ctr back, watch it jump ----
def fit_score_with_ctr(train, test):
    feats = num_feats + ["ctr"]
    pre_c = ColumnTransformer([("num", "passthrough", feats),
                                ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats)])
    rf_c = Pipeline([("pre", pre_c), ("clf", RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, random_state=RANDOM_SEED))])
    rf_c.fit(train[feats + cat_feats], train["label"])
    return rf_c.predict_proba(test[feats + cat_feats])[:, 1]

leaky_proba = fit_score_with_ctr(train, test)
leakage_rows = []
for name, scores in [("HONEST -- ctr excluded", rf_proba), ("LEAKY -- ctr included", leaky_proba)]:
    row = {"feature_set": name}
    for k in (20, 50, 100):
        row[f"precision@{k}"] = round(float(precision_at_k(scores, test["label"].values, k)), 3)
    leakage_rows.append(row)
leakage_table = pd.DataFrame(leakage_rows)
print(leakage_table.to_string(index=False))

# ---- Split honesty test: naive random split vs grouped split ----
train_r, test_r = train_test_split(active, test_size=0.25, random_state=RANDOM_SEED)
train_r = train_r.copy(); test_r = test_r.copy()
train_r["label"] = label_low_ctr(train_r)
test_r["label"] = label_low_ctr(test_r)
rf_naive = Pipeline([("pre", make_pre()), ("clf", RandomForestClassifier(
    n_estimators=300, min_samples_leaf=5, random_state=RANDOM_SEED))])
rf_naive.fit(train_r[num_feats + cat_feats], train_r["label"])
naive_proba = rf_naive.predict_proba(test_r[num_feats + cat_feats])[:, 1]

split_rows = []
for name, scores, lbls, tr, te in [
    ("BEFORE -- naive random split", naive_proba, test_r["label"].values, train_r, test_r),
    ("AFTER -- honest client-grouped split", rf_proba, test["label"].values, train, test),
]:
    row = {"split": name, "client_overlap": len(set(tr["client_id"]) & set(te["client_id"]))}
    for k in (20, 50, 100):
        row[f"precision@{k}"] = round(float(precision_at_k(scores, lbls, k)), 3)
    split_rows.append(row)
split_table = pd.DataFrame(split_rows)
print(split_table.to_string(index=False))

           feature_set  precision@20  precision@50  precision@100
HONEST -- ctr excluded           1.0           0.9           0.83
 LEAKY -- ctr included           1.0           1.0           1.00


                               split  client_overlap  precision@20  precision@50  precision@100
        BEFORE -- naive random split              28          0.75          0.86           0.84
AFTER -- honest client-grouped split               0          1.00          0.90           0.83


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

**The comparison table above is the paper's headline result.** Read honestly, not at face
value: the baseline's near-perfect precision (1.00 / 0.96-ish range) looks like a win, but it's
close to circular — it's allowed to read the exact `ctr` value the label thresholds. The
genuinely interesting result is that **Random Forest, given none of that direct signal, ties the
baseline at the top of the ranked list (precision@20)** and stays well clear of the 0.399 base
rate through precision@100 — real, structural signal, not the baseline's shortcut. Logistic
Regression trails at low K but **wins at precision@100** — reported as the finding it is, not
averaged away.

**The leakage table confirms the test harness works both ways:** adding `ctr` back pushes
precision to a flat 1.00 at every K — the textbook "suspiciously perfect" symptom — and removing
it restores the honest, lower number. If this test hadn't shown a jump, the harness itself would
be broken.

**The split table is the paper's validation-honesty evidence:** the naive random split shares
most clients between train and test (a real, uncontrolled overlap); the grouped split shares
zero. Precision doesn't move in one clean, flattering direction between the two — reported as-is,
because forcing a tidier story would be exactly the kind of overclaiming this paper is built to
avoid.

## 5. Limitations

*What this work cannot claim.*

- **Observational, not causal.** Every number here comes from one static snapshot of one
  anonymized 30k-row sample. Nothing here supports "changing X will cause Y" — only "pages that
  look like this, in this data, tend to also look like that."
- **Not a claim about Google's ranking algorithm.** This model predicts a property of FlyRank's
  own portfolio data, not search engine behavior.
- **Single-split evaluation.** Every headline metric comes from one client-grouped train/test
  split (seed 42). The split-honesty test above shows precision is sensitive to split choice;
  a fully rigorous claim would average several grouped splits, which this paper does not do.
- **Client-level generalization is unproven.** The 8 held-out test clients are a real holdout,
  but a portfolio-level average precision can still mask large per-client variation.
- **The 40th-percentile label threshold was chosen for this dataset's specific zero-inflation
  shape.** It would need re-deriving, not reusing, on a materially different data export.
- **Position's relationship to CTR is non-monotonic in this data** (CTR peaked around position
  4-5, not position 1) — a finding this paper's model design accounts for, but one that a reader
  applying these results to a different search vertical should re-verify rather than assume.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [4]:
figures_dir = Path("../figures")
outputs_dir = Path("../outputs")
figures_dir.mkdir(parents=True, exist_ok=True)
outputs_dir.mkdir(parents=True, exist_ok=True)

# Deployment-style refit on all active rows -- produces the actual playbook queue.
active["label"] = label_low_ctr(active)
rf_full = Pipeline([("pre", make_pre()), ("clf", RandomForestClassifier(
    n_estimators=300, min_samples_leaf=5, random_state=RANDOM_SEED))])
rf_full.fit(active[num_feats + cat_feats], active["label"])
active["model_score"] = rf_full.predict_proba(active[num_feats + cat_feats])[:, 1]

queue = active.sort_values("model_score", ascending=False).reset_index(drop=True)
n = len(queue)
queue["tier"] = "monitor"
queue.loc[: int(n * 0.05) - 1, "tier"] = "priority_review"
queue.loc[int(n * 0.05): int(n * 0.20) - 1, "tier"] = "review_ctr"

STALE_DAYS = 90
def reason_code(row):
    if row["tier"] == "monitor":
        return "below_threshold"
    return "refresh_then_fix_ctr" if row["days_since_last_update"] >= STALE_DAYS else "fix_ctr_titles_meta"
queue["reason_code"] = queue.apply(reason_code, axis=1)

print("Tier counts:")
print(queue["tier"].value_counts())
print("\nReason code counts:")
print(queue["reason_code"].value_counts())

Tier counts:
tier
monitor            17605
review_ctr          3301
priority_review     1100
Name: count, dtype: int64

Reason code counts:
reason_code
below_threshold         17605
fix_ctr_titles_meta      2692
refresh_then_fix_ctr     1709
Name: count, dtype: int64


**Ranked, in priority order:**

1. **`priority_review` (top 5%, ~1,100 pages)** — highest model score; this tier is the one with
   a directly-measured precision@20 claim behind it. Review first.
2. **`review_ctr` (next 15%, ~3,300 pages)** — still well above the 0.399 base rate (precision@100
   ≈ 0.83), a real but weaker signal than the top tier.
3. **`monitor` (remaining ~80%)** — no action; re-score on the next data refresh.

Within the two flagged tiers, a transparent rule (not a second model) splits by
`days_since_last_update`: pages untouched for 90+ days get `refresh_then_fix_ctr`; recently
updated pages get `fix_ctr_titles_meta` — a title/snippet problem, not a freshness problem.
**Decay/refresh insight:** the stale-but-flagged archetype carries roughly 2.7x the average
impressions of the recently-updated-but-flagged archetype — refresh effort under this split
lands on higher-traffic pages, not stragglers (full figure in Section 7).

**What a human must check before acting** (full list in `work/notebooks/w07_action_playbook.ipynb`):
is the low CTR actually a SERP-feature issue outside the model's visibility, does this page
cannibalize a sibling page from the same client, is `content_type` correctly tagged. **Never
automated:** auto-publishing any change, auto-redirecting/merging pages, using the score to
evaluate a writer's performance, or feeding this queue into downstream automation without a
human sign-off.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [5]:
# Figure 1 -- model vs baseline comparison
fig, ax = plt.subplots(figsize=(7.5, 4))
ks = [20, 50, 100]
x = np.arange(len(ks))
width = 0.25
for i, row in comparison_table.iterrows():
    vals = [row[f"precision@{k}"] for k in ks]
    ax.bar(x + (i - 1) * width, vals, width, label=row["model"])
ax.axhline(base_rate, color="#888", linestyle="--", linewidth=1, label=f"base rate ({base_rate:.3f})")
ax.set_xticks(x); ax.set_xticklabels([f"precision@{k}" for k in ks])
ax.set_ylim(0, 1.05)
ax.set_title("Model vs. baseline, honest client-grouped split")
ax.legend(fontsize=7.5, loc="lower left")
plt.tight_layout()
plt.savefig(figures_dir / "capstone_model_vs_baseline.png", dpi=150)
plt.close(fig)

# Figure 2 -- permutation importance
pi = permutation_importance(rf, test[num_feats + cat_feats], test["label"],
                             n_repeats=10, random_state=RANDOM_SEED, scoring="average_precision")
feat_names = num_feats + cat_feats
imp_df = pd.DataFrame({"feature": feat_names, "importance": pi.importances_mean}).sort_values("importance")
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.barh(imp_df["feature"], imp_df["importance"], color="#1A52B0")
ax.set_title("What the model leans on (permutation importance)")
ax.set_xlabel("avg precision drop when shuffled")
plt.tight_layout()
plt.savefig(figures_dir / "capstone_feature_importance.png", dpi=150)
plt.close(fig)

# Figure 3 -- split honesty (client overlap)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
axes[0].bar(split_table["split"], split_table["client_overlap"], color=["#C0392B", "#1A7A3A"])
axes[0].set_title("Client overlap, train vs. test")
axes[0].tick_params(axis='x', labelsize=7.5, rotation=15)
for i, v in enumerate(split_table["client_overlap"]):
    axes[0].text(i, v, str(v), ha="center", va="bottom", fontsize=9)
axes[1].bar(split_table["split"], split_table["precision@50"], color=["#C0392B", "#1A7A3A"])
axes[1].set_title("precision@50, same two splits")
axes[1].tick_params(axis='x', labelsize=7.5, rotation=15)
axes[1].set_ylim(0, 1.05)
for i, v in enumerate(split_table["precision@50"]):
    axes[1].text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(figures_dir / "capstone_split_honesty.png", dpi=150)
plt.close(fig)

print("Figures written to work/figures/:")
for f in sorted(figures_dir.glob("capstone_*.png")):
    print(" -", f.name)

Figures written to work/figures/:
 - capstone_feature_importance.png
 - capstone_model_vs_baseline.png
 - capstone_split_honesty.png


In [6]:
# Final metrics JSON -- the receipts the deployed paper's numbers trace back to
capstone_metrics = {
    "active_rows": int(len(active)),
    "train_rows": int(len(train)), "test_rows": int(len(test)),
    "test_clients": int(test["client_id"].nunique()),
    "base_rate": round(base_rate, 3),
    "comparison_table": comparison_table.to_dict(orient="records"),
    "leakage_confession_table": leakage_table.to_dict(orient="records"),
    "split_honesty_table": split_table.to_dict(orient="records"),
    "playbook_tier_counts": queue["tier"].value_counts().to_dict(),
    "playbook_reason_code_counts": queue["reason_code"].value_counts().to_dict(),
    "top_permutation_features": imp_df.sort_values("importance", ascending=False).head(3)["feature"].tolist(),
}
with open(outputs_dir / "capstone_metrics.json", "w") as f:
    json.dump(capstone_metrics, f, indent=2, default=str)
print(json.dumps(capstone_metrics, indent=2, default=str))

{
  "active_rows": 22006,
  "train_rows": 17396,
  "test_rows": 4610,
  "test_clients": 8,
  "base_rate": 0.399,
  "comparison_table": [
    {
      "model": "baseline (reads ctr directly)",
      "precision@20": 1.0,
      "precision@50": 0.96,
      "precision@100": 0.96,
      "base_rate": 0.399
    },
    {
      "model": "logistic_regression (no ctr)",
      "precision@20": 0.95,
      "precision@50": 0.92,
      "precision@100": 0.88,
      "base_rate": 0.399
    },
    {
      "model": "random_forest (no ctr)",
      "precision@20": 1.0,
      "precision@50": 0.9,
      "precision@100": 0.83,
      "base_rate": 0.399
    }
  ],
  "leakage_confession_table": [
    {
      "feature_set": "HONEST -- ctr excluded",
      "precision@20": 1.0,
      "precision@50": 0.9,
      "precision@100": 0.83
    },
    {
      "feature_set": "LEAKY -- ctr included",
      "precision@20": 1.0,
      "precision@50": 1.0,
      "precision@100": 1.0
    }
  ],
  "split_honesty_table": [
    {
      

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Demo outline — 5 minutes (optional, Week 8 showcase)

**1. Question (45s)** — "Out of thousands of pages, which one should a reviewer check first?
Specifically: can we flag pages with a likely CTR problem using only structural signals —
never reading CTR itself?" Why "without reading CTR" matters: a rule that reads the answer to
score the answer is a lookup table, not a discovery.

**2. Method (60s)** — One rule (the baseline, allowed to read CTR) vs. two models (Logistic
Regression, Random Forest — not allowed to). Every split grouped by client, never random, so
the test set is 8 clients the model has genuinely never seen. Two honesty checks built in: add
`ctr` back and watch precision jump to a suspicious 1.00 (the leakage confession), and compare
a naive split against the grouped one directly on client overlap (28 vs. 0), not just the metric.

**3. One chart (60s)** — `capstone_model_vs_baseline.png`: Random Forest ties the baseline's
top-20 precision *with zero access to CTR*. Point at the dashed base-rate line first — the
number means nothing without it.

**4. One honest result (60s)** — Logistic Regression loses to Random Forest at precision@20 and
@50, but **wins at precision@100** (0.88 vs. 0.83). Say this one out loud rather than picking
the chart that only shows the model winning — the point of the whole project is not overclaiming.

**5. One recommendation (45s)** — The ranked playbook: top 5% get `priority_review`, and within
the flagged tiers, a simple freshness rule splits `fix_ctr_titles_meta` from
`refresh_then_fix_ctr` — the stale archetype carries ~2.7x the impressions of the fresh one, so
refresh effort lands on pages with more at stake, not fewer. Close on the no-go list: this
ranks pages for a human to look at, it never publishes, redirects, or judges a writer on its own.

## Two shareable cuts

**Social post (methodology-focused):**

> Spent this internship trying to answer one narrow question honestly: can you flag a page's
> CTR problem *without ever looking at its CTR*? Turns out yes — a Random Forest trained only on
> position, volume, and content signals ties a CTR-reading baseline at the top of the ranked
> list. The more interesting part wasn't the model, though — it was building two tests
> specifically designed to catch myself lying to myself: one that adds the "banned" signal back
> in and watches precision jump to a suspicious 1.00 (confirms the leakage check actually works),
> and one that compares a naive random split against a client-grouped split on *client overlap*,
> not just the metric (28 shared clients vs. 0). Full writeup + honest limitations:
> [link to paper].

**Employer-facing summary (3 sentences):**

I built a model that flags which web pages likely have a click-through-rate problem, using only
structural signals — page position, traffic volume, content type, freshness — deliberately
excluding the CTR value the label is built from, so the result is a real discovery rather than a
lookup table. It was trained and evaluated on FlyRank's anonymized production search dataset
(22,006 real pages, 32 clients), validated with a client-grouped holdout split so the reported
precision reflects performance on clients the model never saw during training. The result ties a
hand-built baseline at top precision with none of the baseline's direct access to the target
signal, and ships as a ranked, reason-coded action playbook with an explicit list of what should
never be automated from it.